# Narrowing down variant sites causative to stem juiciness phenotype
Assumed to be found on SBC4, 10, and 23 exclusively, based on the association with the phenotype.

In [16]:
# open vcf file
import numpy as np
import cyvcf2
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import gzip, re

sns.set_theme(style="whitegrid")

SAMPLE   = "SBC4, 10, and 23"
VCF_PATH = "../results/sift4g/SBC4_SBC10_SBC23.sift4g.vcf.gz"
GFF_PATH = "../resources/annot/GCF_000003195.3_Sorghum_bicolor_NCBIv3_genomic.gff.gz"

## Overall steps
1. Prepare dataframe
2. Scoring variant sites based on its impact & risk score
3. Convert variant sites into gene using 4 different approaches

These steps is executed by the following script

In [ ]:
!cd .. && ./docker/run.sh python3 workflow/scripts/genomics_scoring_stable.py -i results/sift4g/SBC4_SBC10_SBC23.sift4g.vcf.gz -o /home/daffa/Work/2026/thesis/results/gene_score/ -g resources/annot/GCF_000003195.3_Sorghum_bicolor_NCBIv3_genomic.gff.gz

[genomics_scoring] Parsing VCF: results/sift4g/SBC4_SBC10_SBC23.sift4g.vcf.gz
[genomics_scoring]   205,996 (variant, gene) rows
[genomics_scoring] Loading gene lengths: resources/annot/GCF_000003195.3_Sorghum_bicolor_NCBIv3_genomic.gff.gz
[genomics_scoring]   32,458 genes with length
[genomics_scoring] Written → /home/daffa/Work/2026/thesis/results/gene_score/SBC4_SBC10_SBC23.gene_max.tsv  (12,917 genes)
[genomics_scoring] Written → /home/daffa/Work/2026/thesis/results/gene_score/SBC4_SBC10_SBC23.gene_sum.tsv  (12,917 genes)
[genomics_scoring] Written → /home/daffa/Work/2026/thesis/results/gene_score/SBC4_SBC10_SBC23.gene_max_norm.tsv  (12,917 genes)
[genomics_scoring] Written → /home/daffa/Work/2026/thesis/results/gene_score/SBC4_SBC10_SBC23.gene_sum_norm.tsv  (12,917 genes)
[genomics_scoring] Written → /home/daffa/Work/2026/thesis/results/gene_score/SBC4_SBC10_SBC23.scores_summary.png


## Analysis

In [29]:
df_scored_gene_max      = pd.read_csv("/home/daffa/Work/2026/thesis/results/gene_score/SBC4_SBC10_SBC23.gene_max.tsv", sep="\t")
df_scored_gene_sum      = pd.read_csv("/home/daffa/Work/2026/thesis/results/gene_score/SBC4_SBC10_SBC23.gene_sum.tsv", sep="\t") 
df_scored_gene_max_norm = pd.read_csv("/home/daffa/Work/2026/thesis/results/gene_score/SBC4_SBC10_SBC23.gene_max_norm.tsv", sep="\t")
df_scored_gene_sum_norm = pd.read_csv("/home/daffa/Work/2026/thesis/results/gene_score/SBC4_SBC10_SBC23.gene_sum_norm.tsv", sep="\t")

In [30]:
df_scored_gene_max

,ann_gene_id,ann_gene_name,ann_biotype,chrom,score
0,LOC8059935,LOC8059935,protein_coding,NC_012870.2,1.0
1,LOC8059968,LOC8059968,protein_coding,NC_012870.2,1.0
2,LOC110433559,LOC110433559,pseudogene,NC_012872.2,1.0
3,LOC8082146,LOC8082146,protein_coding,NC_012870.2,1.0
4,LOC8060306,LOC8060306,protein_coding,NC_012872.2,1.0
...,...,...,...,...,...
12912,LOC8057913,LOC8057913,protein_coding,NC_012871.2,0.0
12913,LOC8057932,LOC8057932,protein_coding,NC_012871.2,0.0
12914,LOC8057933,LOC8057933,protein_coding,NC_012871.2,0.0
12915,LOC8057933-LOC8059715,LOC8057933-LOC8059715,NaN,NC_012871.2,0.0


In [31]:
df_scored_gene_max_norm

,ann_gene_id,ann_gene_name,ann_biotype,chrom,score_raw,gene_length,score
0,LOC110435944,LOC110435944,protein_coding,NC_012874.2,1.0,309.0,3.236246
1,LOC110437609,LOC110437609,protein_coding,NC_012877.2,1.0,342.0,2.923977
2,LOC110432712,LOC110432712,protein_coding,NC_012871.2,1.0,372.0,2.688172
3,LOC110434814,LOC110434814,protein_coding,NC_012873.2,1.0,396.0,2.525253
4,LOC8081850,LOC8081850,protein_coding,NC_012870.2,1.0,414.0,2.415459
...,...,...,...,...,...,...,...
12912,LOC8057898-LOC8057899,LOC8057898-LOC8057899,NaN,NC_012876.2,0.0,NaN,NaN
12913,LOC8057902-LOC8057903,LOC8057902-LOC8057903,NaN,NC_012876.2,0.0,NaN,NaN
12914,LOC8057903-LOC8055162,LOC8057903-LOC8055162,NaN,NC_012876.2,0.0,NaN,NaN
12915,LOC8057912-LOC8057913,LOC8057912-LOC8057913,NaN,NC_012871.2,0.0,NaN,NaN


In [32]:
df_scored_gene_sum

,ann_gene_id,ann_gene_name,ann_biotype,chrom,score,n_rows
0,LOC8079158,LOC8079158,protein_coding,NC_012874.2,40.647646,181
1,LOC8064581,LOC8064581,protein_coding,NC_012877.2,30.923417,128
2,LOC110435997,LOC110435997,protein_coding,NC_012874.2,17.070000,47
3,LOC8076751,LOC8076751,protein_coding,NC_012874.2,15.280000,51
4,LOC110436451,LOC110436451,protein_coding,NC_012875.2,14.219594,92
...,...,...,...,...,...,...
12912,LOC8057913,LOC8057913,protein_coding,NC_012871.2,0.000000,2
12913,LOC8057932,LOC8057932,protein_coding,NC_012871.2,0.000000,1
12914,LOC8057933,LOC8057933,protein_coding,NC_012871.2,0.000000,2
12915,LOC8057933-LOC8059715,LOC8057933-LOC8059715,NaN,NC_012871.2,0.000000,1


In [ ]:
df_scored_gene_sum_norm

,ann_gene_id,ann_gene_name,ann_biotype,chrom,score_raw,n_rows,gene_length,score
0,LOC110435944,LOC110435944,protein_coding,NC_012874.2,4.619594,180,309.0,14.950142
1,LOC110435997,LOC110435997,protein_coding,NC_012874.2,17.070000,47,1444.0,11.821330
2,LOC8081850,LOC8081850,protein_coding,NC_012870.2,4.639188,80,414.0,11.205768
3,LOC8064581,LOC8064581,protein_coding,NC_012877.2,30.923417,128,3445.0,8.976318
4,LOC110430402,LOC110430402,protein_coding,NC_012878.2,3.000000,34,429.0,6.993007
...,...,...,...,...,...,...,...,...
12912,LOC8057898-LOC8057899,LOC8057898-LOC8057899,NaN,NC_012876.2,0.000000,13,NaN,NaN
12913,LOC8057902-LOC8057903,LOC8057902-LOC8057903,NaN,NC_012876.2,0.000000,1,NaN,NaN
12914,LOC8057903-LOC8055162,LOC8057903-LOC8055162,NaN,NC_012876.2,0.000000,4,NaN,NaN
12915,LOC8057912-LOC8057913,LOC8057912-LOC8057913,NaN,NC_012871.2,0.000000,2,NaN,NaN
